[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_01/ATS_ch1_calibration_scores/ATS_ch1_calibration_scores.ipynb)

# ATS_ch1_calibration_scores

Calibration and proper scoring rules: PIT histograms of ideal, too sharp, too wide and biased Normal forecasts with their CRPS and log scores; expected log score, CRPS and the improper linear score as functions of the forecast spread (Gneiting, Balabdaoui and Raftery 2007; Gneiting and Raftery 2007).

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 1: Forecast evaluation, scoring rules and combination. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_1'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (Hansen's SPA test) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io

import urllib.error

from scipy import special

import statsmodels.api as sm

SEED = 2026

HICP_RO = ('prc_hicp_minr', 'M.RCH_A.TOTAL.RO')

RATE_RO = ('irt_st_m', 'M.IRT_M3.RO')

RATE_EA = ('irt_st_m', 'M.IRT_M3.EA')

SPF_URL = 'https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/historical-data/SPFmicrodata.xlsx'

RTDSM_URL = 'https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/routput_first_second_third.xlsx'

LOAD_API = 'https://api.energy-charts.info/public_power?country=ro&start={a}&end={b}'

LOAD_YEARS = (2023, 2024, 2025, 2026)

LOAD_END = '2026-09-30'

DGT_SPLIT = ('2000-01-01', '2012-12-31', '2026-09-18')

INFL_H, INFL_WIN, INFL_P = 12, 120, 3

INFL_EVAL = '2013-01-01'

BNR_TARGET = 2.5

AO_WIN, AO_LAGS, AO_EVAL = 60, 4, '1985-01-01'

FX_START, FX_EVAL = '2005-07-01', '2010-01-01'

LOAD_WIN, LOAD_RES, LOAD_EVAL = 364, 182, '2025-01-01'

TAUS = np.arange(1, 100) / 100

SPF_VAR, SPF_H = 'CPI', 6

SPF_EVAL = (1990, 2025)

SPF_WIN, SPF_MIN = 20, 8

MCS_ALPHA, BOOT_B, BLOCK = 0.1, 2000, 7

_FILES = {}

def get_bytes(url):
    """Download a public file once per session (no key)."""
    import hashlib
    import time
    if url in _FILES:
        return _FILES[url]
    cache = os.environ.get('ATS_CACHE')                 # optional local cache folder (not needed in Colab)
    path = os.path.join(cache, hashlib.md5(url.encode()).hexdigest()) if cache else None
    if path and os.path.exists(path):
        _FILES[url] = open(path, 'rb').read()
        return _FILES[url]
    for attempt in range(6):                            # the Energy-Charts API limits the request rate
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (ATS course)'})
            _FILES[url] = urllib.request.urlopen(req, timeout=180).read()
            break
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt == 5:
                raise
            time.sleep(15 * (attempt + 1))
    if path:
        os.makedirs(cache, exist_ok=True)
        open(path, 'wb').write(_FILES[url])
    return _FILES[url]


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def pinball(y, q, tau):
    """Quantile (pinball) loss rho_tau(y - q) = (1{y < q} - tau)(q - y)."""
    return ((y < q).astype(float) - tau) * (q - y)


def interval_score(y, lo, hi, alpha):
    """Interval score of a central (1 - alpha) interval (Gneiting and Raftery 2007): width plus 2/alpha times misses."""
    return (hi - lo) + 2 / alpha * (lo - y) * (y < lo) + 2 / alpha * (y - hi) * (y > hi)


def crps_normal(y, mu, sigma):
    """CRPS of N(mu, sigma^2) at y (closed form)."""
    z = (y - mu) / sigma
    return sigma * (z * (2 * stats.norm.cdf(z) - 1) + 2 * stats.norm.pdf(z) - 1 / np.sqrt(np.pi))


def crps_t(y, mu, s, nu):
    """CRPS of a location-scale Student t (location mu, scale s, nu > 1 degrees of freedom) at y (closed form)."""
    z = (y - mu) / s
    term = (2 * np.sqrt(nu) / (nu - 1)) * special.beta(0.5, nu - 0.5) / special.beta(0.5, nu / 2) ** 2
    return s * (z * (2 * stats.t.cdf(z, nu) - 1) + 2 * stats.t.pdf(z, nu) * (nu + z ** 2) / (nu - 1) - term)


def hac_var(x, lags, kernel='bartlett', center=True):
    """Long-run variance of a series: Bartlett (Newey-West) or rectangular (truncated) kernel."""
    x = np.asarray(x, float)
    x = x - x.mean() if center else x
    T = len(x)
    v = x @ x / T
    for k in range(1, lags + 1):
        w = 1 - k / (lags + 1) if kernel == 'bartlett' else 1.0
        v += 2 * w * (x[k:] @ x[:-k]) / T
    return v


def dm_test(d, h=1, kernel='rect', lags=None):
    """Diebold-Mariano test of E[d] = 0 for the loss differential d (model 1 minus model 2) with the HAC variance
    (rectangular kernel with h - 1 lags as in Diebold and Mariano 1995; Bartlett if it is negative) and the
    Harvey-Leybourne-Newbold (1997) small-sample correction with Student t(T - 1) critical values."""
    d = np.asarray(d, float)
    d = d[~np.isnan(d)]
    T = len(d)
    L = h - 1 if lags is None else lags
    v = hac_var(d, L, kernel)
    used = kernel
    if v <= 0:
        v, used = hac_var(d, L, 'bartlett'), 'bartlett'
    dm = d.mean() / np.sqrt(v / T)
    k = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    hln = k * dm
    return {'T': T, 'dbar': float(d.mean()), 'dm': float(dm), 'p_dm': float(2 * stats.norm.sf(abs(dm))),
            'hln': float(hln), 'p_hln': float(2 * stats.t.sf(abs(hln), T - 1)), 'kernel': used}


def gw_test(d, Z, h=1):
    """Giacomini-White (2006) test of conditional equal predictive ability: E[Z_t d_{t+h}] = 0, Wald statistic
    T m' Omega^-1 m ~ chi2(q), with Omega the (uncentred) HAC covariance of Z_t d_{t+h} (h - 1 Bartlett lags)."""
    d = np.asarray(d, float)
    Z = np.column_stack([np.ones(len(d))] + [np.asarray(z, float) for z in Z])
    ok = ~np.isnan(d) & ~np.isnan(Z).any(axis=1)
    X = Z[ok] * d[ok, None]
    T, q = X.shape
    m = X.mean(axis=0)
    S = X.T @ X / T
    for k in range(1, h):
        G = X[k:].T @ X[:-k] / T
        S += (1 - k / h) * (G + G.T)
    stat = float(T * m @ np.linalg.solve(S, m))
    return {'T': int(T), 'q': int(q), 'stat': stat, 'p': float(stats.chi2.sf(stat, q))}


def cw_test(y, f_small, f_big, h=1):
    """Clark-West (2007) test for nested models: adjusted MSPE differential
    f_t = e_small^2 - [e_big^2 - (f_small - f_big)^2]; one-sided t test (H1: the larger model is better)."""
    e1, e2 = y - f_small, y - f_big
    f = e1 ** 2 - (e2 ** 2 - (f_small - f_big) ** 2)
    f = np.asarray(f, float)
    T = len(f)
    v = hac_var(f, max(h - 1, 0), 'bartlett')
    t = f.mean() / np.sqrt(v / T)
    return {'T': T, 'cw': float(t), 'p': float(stats.norm.sf(t))}


def mz_test(y, f, h=1):
    """Mincer-Zarnowitz regression y = a + b f + u with Newey-West standard errors (h - 1 lags, at least 1)
    and the Wald test of (a, b) = (0, 1)."""
    import statsmodels.api as sm
    X = sm.add_constant(np.asarray(f, float))
    m = sm.OLS(np.asarray(y, float), X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h - 1, 1)})
    w = m.wald_test((np.eye(2), np.array([0.0, 1.0])), scalar=True)
    return {'a': float(m.params[0]), 'b': float(m.params[1]), 'se_a': float(m.bse[0]), 'se_b': float(m.bse[1]),
            'wald': float(w.statistic), 'p': float(w.pvalue), 'r2': float(m.rsquared)}


def encompassing_test(y, f1, f2, h=1):
    """Harvey-Leybourne-Newbold (1998) test of H0: forecast 1 encompasses forecast 2, d_t = e1_t (e1_t - e2_t),
    one-sided (H1: E d > 0, forecast 2 adds information), HLN-corrected statistic with t(T - 1)."""
    e1, e2 = y - f1, y - f2
    r = dm_test(e1 * (e1 - e2), h)
    r['p_one'] = float(stats.t.sf(r['hln'], r['T'] - 1))
    return r


def berkowitz_test(u):
    """Berkowitz (2001) LR test: z = Phi^-1(PIT) is i.i.d. N(0, 1) against an AR(1) with free mean and variance."""
    z = stats.norm.ppf(np.clip(np.asarray(u, float), 1e-10, 1 - 1e-10))
    y, x = z[1:], z[:-1]
    X = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ b
    s2 = res @ res / len(y)
    ll1 = np.sum(stats.norm.logpdf(res, scale=np.sqrt(s2)))
    ll0 = np.sum(stats.norm.logpdf(y))
    lr = float(2 * (ll1 - ll0))
    return {'mu': float(b[0] / (1 - b[1])), 'rho': float(b[1]), 'sigma': float(np.sqrt(s2 / (1 - b[1] ** 2))),
            'lr': lr, 'p': float(stats.chi2.sf(lr, 3))}


def mcs(losses, alpha=MCS_ALPHA, B=BOOT_B, block=BLOCK, seed=SEED):
    """Model Confidence Set (Hansen, Lunde and Nason 2011) with the T_max statistic and a moving-block bootstrap:
    eliminate the model with the largest standardised loss differential until the equivalence test is not rejected.
    losses: T x m DataFrame. Returns the MCS p-value of every model and the set at level alpha."""
    L = np.asarray(losses, float)
    L = L[~np.isnan(L).any(axis=1)]
    T, m = L.shape
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    idx = np.concatenate([np.arange(s, s + block) for s in rng.integers(0, T - block + 1, size=(B, nb)).ravel()])
    idx = idx.reshape(B, nb * block)[:, :T]
    Lb = L[idx].mean(axis=1)                          # B x m bootstrap means
    alive = list(range(m))
    pvals = {}
    p_run = 0.0
    while len(alive) > 1:
        Lm = L[:, alive].mean(axis=0)
        dbar = Lm - Lm.mean()                         # d_i. = loss of i minus the average loss of the set
        db = Lb[:, alive] - Lb[:, alive].mean(axis=1, keepdims=True)
        se = np.sqrt(((db - dbar) ** 2).mean(axis=0))
        t = dbar / se
        tb = ((db - dbar) / se).max(axis=1)
        p = float((tb > t.max()).mean())
        p_run = max(p_run, p)
        worst = alive[int(np.argmax(t))]
        pvals[worst] = p_run
        alive.remove(worst)
    pvals[alive[0]] = 1.0
    cols = list(losses.columns) if hasattr(losses, 'columns') else list(range(m))
    out = {cols[i]: pvals[i] for i in range(m)}
    return {'p': out, 'set': [c for c in cols if out[c] >= alpha]}


def fig_pit_shapes(save_it=True, n=5000):
    """PIT histograms of four Normal forecasts of Y ~ N(0, 1): ideal, too sharp, too wide, biased."""
    rng = np.random.default_rng(SEED)
    y = rng.standard_normal(n)
    cases = [('ideal: N(0, 1)', 0, 1, st.MainBlue), ('too sharp: N(0, 0.6^2)', 0, 0.6, st.IDAred),
             ('too wide: N(0, 1.6^2)', 0, 1.6, st.Forest), ('biased: N(0.5, 1)', 0.5, 1, st.Purple)]
    fig, axs = plt.subplots(1, 4, figsize=(12, 3.0), sharey=True)
    out = {}
    for ax, (lab, m, s, c) in zip(axs, cases):
        u = stats.norm.cdf(y, m, s)
        ax.hist(u, bins=20, range=(0, 1), density=True, color=c, alpha=0.85, label=lab)
        ax.axhline(1, color=st.DarkText, lw=0.8, ls='--')
        ax.set_xlabel('PIT')
        st.legend_outside_bottom(ax, ncol=1, y=-0.28)
        out[lab.split(':')[0]] = {'share_tails': float(np.mean((u < 0.05) | (u > 0.95))),
                                  'crps': float(np.mean(crps_normal(y, m, s))),
                                  'logs': float(-np.mean(stats.norm.logpdf(y, m, s)))}
    axs[0].set_ylabel('density')
    plt.tight_layout()
    save('ats_ch1_pit_shapes', save_it)
    return out


def fig_proper_scores(save_it=True):
    """Expected scores of N(0, s^2) forecasts when Y ~ N(0, 1), as functions of s: log score and CRPS are minimised
    at s = 1 (proper); the linear score -f(y) rewards s -> 0 (improper)."""
    s = np.linspace(0.3, 2.5, 221)
    logs = 0.5 * np.log(2 * np.pi * s ** 2) + 1 / (2 * s ** 2)          # E[-log f(Y)]
    crps = np.sqrt(1 + s ** 2) * np.sqrt(2 / np.pi) - s / np.sqrt(np.pi)  # E CRPS(N(0, s^2), Y), Y ~ N(0, 1)
    lin = -1 / np.sqrt(2 * np.pi * (1 + s ** 2))                          # E[-f(Y)]
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
    for ax, v, c, lab in [(axs[0], logs, st.MainBlue, 'expected log score'), (axs[1], crps, st.Forest, 'expected CRPS'),
                          (axs[2], lin, st.IDAred, 'expected linear score -f(y)')]:
        ax.plot(s, v, color=c, lw=2, label=lab)
        ax.axvline(1, color=st.Amber, ls='--', lw=1.2, label='true spread s = 1')
        ax.set_xlabel('forecast spread s')
        st.legend_outside_bottom(ax, ncol=1, y=-0.27)
    axs[0].set_ylabel('expected score')
    plt.tight_layout()
    save('ats_ch1_proper_scores', save_it)
    return {'argmin_logs': float(s[logs.argmin()]), 'argmin_crps': float(s[crps.argmin()]), 'argmin_lin': float(s[lin.argmin()])}

## Run

In [ ]:
print(fig_pit_shapes())
print(fig_proper_scores())

![ats_ch1_pit_shapes](./ats_ch1_pit_shapes.png)

Output: `ats_ch1_pit_shapes.pdf`

![ats_ch1_proper_scores](./ats_ch1_proper_scores.png)

Output: `ats_ch1_proper_scores.pdf`